# PAFA 仓库 BEATs+CE 复现：seed 42

本 Notebook 是零执行 orchestration entrypoint，研究问题是复现 PAFA 作者仓库中的 **BEATs+CE baseline**，不是 PAFA loss。任务固定为 ICBHI single-source、native respiratory-cycle unit、flat4（Normal/Crackle/Wheeze/Both）direct softmax CE。

Verified source contract：作者 `scripts/beats_ce.sh` 使用 BEATs iter3+ AS2M full fine-tuning、16 kHz、5 s repeat-pad/front-truncate、batch 32、Adam lr=5e-5、weight decay=1e-6、100 epochs、cosine、EMA beta=0.5、`--nospec`。本入口复用作者 BEATs model/frontend，并保存每 epoch checkpoint 与 selection predictions。


## 两种证据模式（目录与结论严格隔离）

- **Mode A / `author_test_selected`**：按作者代码每 epoch 读取 official test，以 `Score=(Sp+Se)/2` 选 best。只能标记为 `test_selected_reproduction_receipt`，不能称 clean estimate。
- **Mode B / `clean_validation_only`**：selection phase 只读取 official-train annotations、只解码 official-train audio，并使用固定 split seed 42 的 patient-grouped 5-fold fold 0 作为 validation；selected checkpoint 固定后才首次读取 official-test annotations、解码 official-test audio、构建 terminal loader并评测一次。官方 recording split 本身仍不是 strict patient-held-out。

Mode A 与 Mode B 分别写入 `result/reproduce/pafa_beats_ce/author_test_selected/seed_42/` 和 `result/reproduce/pafa_beats_ce/clean_validation_only/seed_42/`，禁止覆盖或互相继承 selection。seed 42 是首条 local run；未来作者 1–5 seeds 通过同一 `--seed` 接口扩展，但本轮不运行。


In [ ]:
import os
from pathlib import Path

EXECUTE = False  # 只有收到服务器正式启动批准后才能改为 True。
SEED = 42
FUTURE_AUTHOR_SEEDS = [1, 2, 3, 4, 5]
AUDIO_DIR = Path(os.environ.get(
    'ICBHI_AUDIO_DIR',
    'dataset/raw/icbhi_2017/source_original/ICBHI_final_database/ICBHI_final_database',
))
AUTHOR_REPO = Path(os.environ.get('PAFA_AUTHOR_REPO', '.cache/pafa_beats_ce/source/repo'))
CHECKPOINT = Path(os.environ.get(
    'BEATS_AS2M_CHECKPOINT',
    '.cache/pafa_beats_ce/checkpoints/BEATs_iter3_plus_AS2M.pt',
))
OUTPUT_BASE = Path('result/reproduce/pafa_beats_ce')

common = [
    'python', '-m', 'baseline.pafa.beats_ce_reproduction',
    '--seed', str(SEED),
    '--audio-dir', str(AUDIO_DIR),
    '--author-repo', str(AUTHOR_REPO),
    '--checkpoint', str(CHECKPOINT),
    '--output-base', str(OUTPUT_BASE),
    '--device', 'cuda',
]
author_test_selected_command = common + ['--mode', 'author_test_selected']
clean_validation_only_command = common + ['--mode', 'clean_validation_only']
commands = [author_test_selected_command, clean_validation_only_command]
commands


In [ ]:
if EXECUTE:
    import subprocess
    for command in commands:
        subprocess.run(command, check=True)


## 输出合同与 claim boundary

每个 mode 保存：`config.json`、cycle-level split/lineage、100 个 epoch checkpoints、每 epoch selection predictions、`selection_log.jsonl`、selected checkpoint 与 `run_summary.json`。Clean mode 的 selection split 文件不含 test record；terminal test lineage 只在 selection 完成后写入 `terminal/official_test_split.jsonl`。Predictions 至少包含 stable sample ID、recording filename、patient/group、ground truth、logits、probabilities、prediction；metrics 包含 Sp、Se、Score、macro-F1、UAR、support、per-class recall 与 confusion。

磁盘预算：代码、数据和 pretrained checkpoint 均直接复用。完整 BEATs 权重只在 selection Score 改善时覆盖保存为 best_checkpoint.pt，不保存 Adam state；每个 mode 建议预留 2 GB。每个 epoch 仍保留小型 predictions 与 metrics，因此可以审计 selection curve，但不能在训练后恢复任意 epoch 权重或无损恢复 optimizer。

Mode A 的 test 是 selection-exposed，任何数字都必须带 `test-selected reproduction`；Mode B 才是 validation-only main。两者均为 seed-42 local reproduction，不等于论文五种子均值。作者仓库没有 LICENSE、未发布 task checkpoint；这里只使用 pretrained BEATs package，不推断新的许可结论。

**Test Result: Not run. Decision: READY_FOR_SERVER_APPROVAL；本地未启动 model forward、训练、validation 或 test。**
